In [3]:
from pathlib import Path
import re
import datetime

# ===== Experiment setup =====
MODEL = "llama3.3-70b"
TEMPERATURE = 0
CONFIG_NAME = "C-Min"

scenario_name = "Car manufacturing supply chain management"


def slugify(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return re.sub(r"_+", "_", s).strip("_")

# ===== Create log file =====
logs_dir = Path("logs")
logs_dir.mkdir(exist_ok=True)

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
txt_path = logs_dir / f"{slugify(scenario_name)}__{slugify(CONFIG_NAME)}__{stamp}.txt"

with txt_path.open("w", encoding="utf-8") as f:
    f.write("EXPERIMENT LOG\n")
    f.write("=" * 70 + "\n")
    f.write(f"SCENARIO: {scenario_name}\n")

    f.write(f"CONFIG: {CONFIG_NAME}\n")
    f.write(f"MODEL: {MODEL}\n")
    f.write(f"TEMPERATURE: {TEMPERATURE}\n")
    f.write(f"CREATED_AT: {stamp}\n")
    f.write("=" * 70 + "\n\n")

print("Log file created:", txt_path)


Log file created: logs/car_manufacturing_supply_chain_management__c_min__20260210-093953.txt


In [4]:
from datetime import datetime; print("Log time:", datetime.now().isoformat(timespec="seconds"))

Log time: 2026-02-10T09:39:56


In [5]:
def dict_to_bullets(d, indent: int = 4) -> str:
    """
    Turn a nested dict (and lists) into indented bullet lines.

    Change vs previous version:
    - FIRST-LEVEL keys are NOT printed (e.g., 'arch', 'relation', 'cara' won't appear).
    - Content under each first-level key is printed directly.
    """
    pad = " " * indent
    lines = []

    if d is None or d == {}:
        return f"{pad}- (none)"

    def walk(node, level: int, show_key: bool = True):
        pad2 = " " * level

        if isinstance(node, dict):
            if not node:
                lines.append(f"{pad2}- (none)")
                return
            for k, v in node.items():
                # Skip printing the first-level key labels
                if show_key:
                    key = str(k) if k is not None and str(k).strip() else "(unnamed)"
                    lines.append(f"{pad2}- {key}:")
                    walk(v, level + 2, show_key=True)
                else:
                    # First level: don't print key, just descend into its value
                    walk(v, level, show_key=True)

        elif isinstance(node, (list, tuple, set)):
            node = list(node)
            if not node:
                lines.append(f"{pad2}- (none)")
            else:
                for item in node:
                    if isinstance(item, (dict, list, tuple, set)):
                        lines.append(f"{pad2}-")
                        walk(item, level + 2, show_key=True)
                    else:
                        lines.append(f"{pad2}- {item}")

        else:
            val = str(node).strip()
            lines.append(f"{pad2}- {val if val else '(empty)'}")

    # Start at root with show_key=False so first-level keys are hidden
    walk(d, indent, show_key=False)
    return "\n".join(lines)


In [6]:
iteration_id=0

In [12]:
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local-dummy")

def generate_completion(
    scenario_name: str,
    elements: dict,
    model: str,
     dsl: str, 
    temperature: float = 0.0,
    n_additions: int = 3,
  
   
) -> tuple[str, str]:
    

    # Build partial model text from elements
    elems_block =  dict_to_bullets(elements, indent=4)
    
    partial_model = f"""Elements: \n {elems_block}"""

    prompt = f"""
                You are a modeling assistant. Complete a partial {dsl} for the scenario.

                Scenario: {scenario_name}
                Current partial model:
                {partial_model}

                Task:
                Propose up to {n_additions} additions to extend the model. An addition could be architectural element ), a relation ), or a characteristic.

                Write as a bullet list. Be concise.
                """.strip()

    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=400,  
    )

    output_text = resp.choices[0].message.content
    return prompt, output_text






In [13]:
iteration_id += 1

prompt, output_text = generate_completion(
    scenario_name=scenario_name,
elements={
    "process": "engine assembly",
}


,
    model=MODEL,
    temperature=TEMPERATURE,
    n_additions=3,
    dsl="supply chain management (SCM) modeling language",

)
print("type(output_text) =", type(output_text))
print("preview(repr) =", repr(output_text)[:300])

print("===== PROMPT =====")
print(prompt)
print("\n===== MODEL OUTPUT =====")
print(output_text)


type(output_text) = <class 'str'>
preview(repr) = ' innrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrs Gongrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrs Gongrsrsrsrsrsrsrsrsrsrs etrsrsrsrs etrsrsrs belrs Gongrsrsrsrs Gongr
===== PROMPT =====
You are a modeling assistant. Complete a partial supply chain management (SCM) modeling language for the scenario.

                Scenario: Car manufacturing supply chain management
                Current partial model:
                Elements: 
     - engine assembly

                Task:
                Propose up to 3 additions to extend the model. An addition could be architectural element ), a relation ), or a characteristic.

                Write as a bullet list. Be concise.

===== MODEL OUTPUT =====
 innrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsrsr

In [ ]:
# ===== Manually fill this dict (we copy/paste items from output) =====
classified = {
    "arch": {
       "Entity": ["engine component",  ],
       "Resource": ["Assembly workstation"],
       "Relationship": []
       

    },
    "connections": [
        "engine component  - engine assembly",
    ],
    "characteristics": [
 
    ],
}

# ===== Log everything (prompt + output + youand the  manual dict we accepted to integrate) =====
with txt_path.open("a", encoding="utf-8") as f:
    f.write(f"\n\nITERATION {iteration_id}\n")
    f.write("-" * 80 + "\n")

    f.write("PARTIAL MODEL\n")
    f.write("-" * 80 + "\n")
    f.write(partial_model.strip() + "\n\n")

    f.write("PROMPT\n")
    f.write("-" * 80 + "\n")
    f.write(prompt + "\n\n")

    f.write("MODEL OUTPUT\n")
    f.write("-" * 80 + "\n")
    f.write(output_text.strip() + "\n\n")

    f.write("MANUAL CLASSIFICATION\n")
    f.write("-" * 80 + "\n")
    f.write("ARCH:\n")
    for x in classified["arch"]:
        f.write(f"- {x}\n")
    f.write("\nCONNECTIONS:\n")
    for x in classified["connections"]:
        f.write(f"- {x}\n")
    f.write("\nCHARACTERISTICS:\n")
    for x in classified["characteristics"]:
        f.write(f"- {x}\n")

print(f"\nLogged iteration {iteration_id} to:", txt_path)
classified


Logged iteration 1 to: logs/medical_appointment_platform__c_min__20260203-111720.txt


{'arch': ['Appointment'],
 'connections': ['user - appointment'],
 'characteristics': ['username']}

In [9]:
# ===== Manually fill accepted items for this iteration =====
accepted = {
    "arch": [
        "Appointment",
    ],
    "connections": [
    
    ],
    "characteristics": [
       "username"
    ],
}

# ===== Append accepted to the same log file =====
with txt_path.open("a", encoding="utf-8") as f:
    f.write("\n\nACCEPTED (ITERATION {})\n".format(iteration_id))
    f.write("-" * 80 + "\n")

    f.write("ARCH:\n")
    for x in accepted["arch"]:
        f.write(f"- {x}\n")

    f.write("\nCONNECTIONS:\n")
    for x in accepted["connections"]:
        f.write(f"- {x}\n")

    f.write("\nCHARACTERISTICS:\n")
    for x in accepted["characteristics"]:
        f.write(f"- {x}\n")

print(f"Accepted suggestions appended for iteration {iteration_id} to:", txt_path)
accepted


Accepted suggestions appended for iteration 1 to: logs/medical_appointment_platform__c_min__20260203-111720.txt


{'arch': ['Appointment'], 'connections': [], 'characteristics': ['username']}